# AI-Based Factory Layout Optimizer — Google Colab Training & Demonstration
**Work System Design Course Project**  
**Title**: *Using Computer Vision to Design a Factory with Optimal Placement of Equipment*  

This notebook trains a PyTorch Faster R-CNN object detector for factory equipment and exports artifacts compatible with the local Streamlit application.

## 1. Environment Setup & Dependency Installation

In [ ]:
# Install dependencies in Google Colab
!pip install -q torch torchvision pillow opencv-python-headless matplotlib pandas scipy plotly

import torch
import torchvision
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Equipment Classes & Synthetic Data Generation

In [ ]:
import os
import json
import random
from pathlib import Path
from PIL import Image, ImageDraw

# Canonical 6 equipment categories (+ 0 for background in TorchVision Faster R-CNN)
CLASS_ID_TO_NAME = {
    0: "__background__",
    1: "Cutting machine",
    2: "Drilling machine",
    3: "Assembly workstation",
    4: "Inspection station",
    5: "Packaging station",
    6: "Storage area"
}

CLASS_COLORS = {
    1: (231, 76, 60),    # Red
    2: (230, 126, 34),   # Orange
    3: (46, 204, 113),   # Green
    4: (155, 89, 182),   # Purple
    5: (243, 156, 18),   # Yellow-Orange
    6: (52, 152, 219),   # Blue
}

def generate_colab_dataset(base_dir="dataset", train_n=25, val_n=5, test_n=5):
    base = Path(base_dir)
    base.mkdir(parents=True, exist_ok=True)
    categories = [{"id": k, "name": v} for k, v in CLASS_ID_TO_NAME.items() if k > 0]
    
    for split, count in [("train", train_n), ("val", val_n), ("test", test_n)]:
        s_dir = base / split
        s_dir.mkdir(parents=True, exist_ok=True)
        images, annotations = [], []
        ann_id = 1
        
        for img_id in range(1, count + 1):
            W, H = 800, 600
            img = Image.new("RGB", (W, H), (235, 237, 240))
            draw = ImageDraw.Draw(img)
            
            # Floor grid
            for x in range(0, W, 40):
                draw.line([(x, 0), (x, H)], fill=(222, 225, 230))
            for y in range(0, H, 40):
                draw.line([(0, y), (W, y)], fill=(222, 225, 230))
            draw.rectangle([0, 0, W-1, H-1], outline=(52, 73, 94), width=8)
            
            # Random non-overlapping equipment boxes
            boxes = []
            for _ in range(random.randint(6, 10)):
                cid = random.randint(1, 6)
                bw, bh = random.randint(70, 110), random.randint(50, 90)
                for _ in range(100):
                    bx = random.randint(30, W - bw - 30)
                    by = random.randint(30, H - bh - 30)
                    box = [bx, by, bx + bw, by + bh]
                    if not any(not (box[2] < b[0] or box[0] > b[2] or box[3] < b[1] or box[1] > b[3]) for b in boxes):
                        boxes.append(box)
                        col = CLASS_COLORS[cid]
                        draw.rectangle(box, fill=col, outline=(44, 62, 80), width=2)
                        draw.text((bx+4, by+4), CLASS_ID_TO_NAME[cid][:8], fill=(255, 255, 255))
                        annotations.append({
                            "id": ann_id, "image_id": img_id, "category_id": cid,
                            "bbox": [bx, by, bw, bh], "area": bw * bh, "iscrowd": 0
                        })
                        ann_id += 1
                        break
            
            img_file = f"{split}_{img_id:04d}.png"
            img.save(s_dir / img_file)
            images.append({"id": img_id, "file_name": img_file, "width": W, "height": H})
            
        coco_dict = {"categories": categories, "images": images, "annotations": annotations}
        with open(s_dir / "_annotations.coco.json", "w") as f:
            json.dump(coco_dict, f, indent=2)
            
    print("Generated Colab dataset successfully!")

generate_colab_dataset()

## 3. PyTorch Model Definition (Faster R-CNN with MobileNetV3 FPN)

In [ ]:
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_model(num_classes=7):
    weights = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
    model = fasterrcnn_mobilenet_v3_large_fpn(weights=weights)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = get_model(num_classes=7).to(device)
print(f"Model instantiated on {device}")

## 4. Training Loop

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

class ColabFactoryDataset(Dataset):
    def __init__(self, s_dir):
        self.s_dir = Path(s_dir)
        with open(self.s_dir / "_annotations.coco.json") as f:
            self.coco = json.load(f)
        self.images = self.coco["images"]
        self.img_to_anns = {}
        for a in self.coco["annotations"]:
            self.img_to_anns.setdefault(a["image_id"], []).append(a)
            
    def __len__(self):
        return len(self.images)
        
    def __getitem__(self, idx):
        info = self.images[idx]
        img = Image.open(self.s_dir / info["file_name"]).convert("RGB")
        anns = self.img_to_anns.get(info["id"], [])
        boxes, labels = [], []
        for a in anns:
            x, y, w, h = a["bbox"]
            boxes.append([x, y, x + w, y + h])
            labels.append(a["category_id"])
        
        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0,4)),
            "labels": torch.as_tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64)
        }
        return TF.to_tensor(img), target

train_ds = ColabFactoryDataset("dataset/train")
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=lambda b: tuple(zip(*b)))

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005)

print("Training Faster R-CNN for 5 epochs...")
model.train()
for epoch in range(1, 6):
    total_loss = 0
    for imgs, tgts in train_loader:
        imgs = [i.to(device) for i in imgs]
        tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]
        loss_dict = model(imgs, tgts)
        loss = sum(l for l in loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}/5 - Avg Loss: {total_loss / len(train_loader):.4f}")

# Save checkpoint
os.makedirs("models", exist_ok=True)
torch.save({"model_state_dict": model.state_dict(), "architecture": "fasterrcnn_mobilenet_v3_large_fpn"}, "models/best_detector.pth")
print("Saved weights to models/best_detector.pth")

## 5. Download Model Checkpoint for Local Streamlit App
Run the cell below in Google Colab to download `best_detector.pth`. Place it into your local `factory-layout-optimizer/models/` directory.

In [ ]:
try:
    from google.colab import files
    files.download("models/best_detector.pth")
    print("Download prompt triggered!")
except Exception as e:
    print(f"Running outside Google Colab: weights are saved locally at models/best_detector.pth ({e})")